# Capstone — Refresh / Content Opportunity Scoring (Lane 2)
Paper mirror for `docs/index.html`. Starter slice only, seed 42. Run top to bottom.

In [ ]:
%pip install -q pandas scikit-learn numpy matplotlib

## 1. Question
Can trailing signals rank pages for refresh review? Decision: review order. Output: score + reason codes. Errors: FP wastes edits, FN lets decay slide.

## 2. Data
`data/raw/content_refresh_anonymized.csv` — 30k rows, 44 cols, 32 clients. Label `is_declining_label=(trend_direction=='down')`, base rate ~0.542. Features exclude `trend_direction`,`trend_pct`,`content_id`,`client_id`.

In [ ]:
import pandas as pd, numpy as np
import sys; sys.path.insert(0,'../../../scripts')
from ml_utils import precision_at_k, MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES
raw=pd.read_csv('../../../data/raw/content_refresh_anonymized.csv')
print(raw.shape, raw['client_id'].nunique(), round((raw['trend_direction']=='down').mean(),4))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
df=raw.copy()
df['is_declining_label']=(df['trend_direction'].str.lower()=='down').astype(int)
df['log_impressions_90d']=np.log1p(df['impressions_90d'])
df['log_clicks_90d']=np.log1p(df['clicks_90d'])
df['log_sessions_90d']=np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d']=np.log1p(df['ai_sessions_90d'])
for c in MODEL_NUMERIC_FEATURES:
    df[c]=df[c].fillna(0)
for c in MODEL_CATEGORICAL_FEATURES:
    df[c]=df[c].fillna('unknown')
    df[c]=LabelEncoder().fit_transform(df[c].astype(str))
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
tr,te=next(gss.split(df,groups=raw['client_id']))
dtr,dte=df.iloc[tr].copy(),df.iloc[te].copy()
assert not set(raw.iloc[tr]['client_id']).intersection(set(raw.iloc[te]['client_id']))
print(f"train {len(dtr)} ({raw.iloc[tr]['client_id'].nunique()}) test {len(dte)} ({raw.iloc[te]['client_id'].nunique()})")

## 3. Methodology
RF(100, depth 10). Baseline `(days_since_last_update>=180)*impressions_90d`. Grouped holdout primary; random split as over-optimism contrast.

In [ ]:
feats=MODEL_NUMERIC_FEATURES+MODEL_CATEGORICAL_FEATURES
dte['baseline_score']=(dte['days_since_last_update']>=180).astype(int)*raw.iloc[te]['impressions_90d'].values
m=RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42,n_jobs=-1)
m.fit(dtr[feats],dtr['is_declining_label'])
dte['model_score']=m.predict_proba(dte[feats])[:,1]
print('AUC base %.3f model %.3f' % (roc_auc_score(dte['is_declining_label'],dte['baseline_score']), roc_auc_score(dte['is_declining_label'],dte['model_score'])))
print('P50 base %.3f model %.3f' % (precision_at_k(dte['is_declining_label'],dte['baseline_score'],50), precision_at_k(dte['is_declining_label'],dte['model_score'],50)))
print('P200 base %.3f model %.3f' % (precision_at_k(dte['is_declining_label'],dte['baseline_score'],200), precision_at_k(dte['is_declining_label'],dte['model_score'],200)))

## 4. Results
Grouped AUC ~0.61 vs 0.50; P50 0.56 vs 0.70 (rule wins head); P200 tie. Random contrast ~0.77.

In [ ]:
imp=pd.DataFrame({'feature':feats,'importance':m.feature_importances_}).sort_values('importance',ascending=False)
print(imp.head(8).to_string(index=False))

## 5. Limitations
Observed/directional/decision-support. New pages unscorable. Intent-blind. Seasonal. Slice-bound.

## 6. Ranked recommendations
Score + codes: STALE_CONTENT, STRIKING_DISTANCE, LOW_CTR_HIGH_VIS, LOW_ENGAGEMENT, MODEL_DECLINE_RISK. Top-50 monthly human review; monitor low-vol/fresh; no autopublish brand/legal.

## 7. Reproducibility
`pip install -r requirements.txt`, Run All, seed 42. Receipts `work/outputs/playbook_metrics.json`.

## Acknowledgments
Built on the FlyRank ML Internship dataset — https://flyrank.ai

## ML-12 — Demo and Summary
**Demo (5m):** Q45s / method1m / chart45s (0.61 honest vs 0.77 random) / honest result1m / playbook1m.
**Social:** Random 0.77, grouped 0.61 — shipped honest number + human playbook. Lower honest beats higher misleading. #ML #DataScience #MLOps
**Employer:** RF refresh ranker on 30k pages; grouped 0.61 vs 0.50, P50 0.56 vs 0.70; reason-coded queue + deployed paper.

## Self-check
- [x] runs top-bottom, no client data, careful words, under work/notebooks, paper 9 sections, ML-12 present.